# Level 6: AI-Assisted Scientific Programming, Reproducibility, Testing, and Presentation

**Course:** ICS 2207 — Scientific Computing  
**Project:** HydroSense-Kenya  
**Objective:** Demonstrate responsible AI use, reproducible workflows, validation, and scientific communication.

---

## 1. AI-Assisted Programming Tasks

We used AI tools (GitHub Copilot and ChatGPT) for **five specific support tasks**:

1. **Problem statement drafting** — AI generated an initial draft; we rewrote statistics and added project-specific equations
2. **Test case generation** — AI scaffolded pytest tests; we adjusted tolerances and added domain-specific cases
3. **README documentation** — AI structured the layout; we corrected paths and added technical details
4. **Code docstrings** — AI generated NumPy-style docstrings; we corrected mathematical descriptions
5. **Data cleaning strategy** — AI suggested approaches for specific anomalies; we implemented with proper detection functions

Full details are in [`AI_USE_LOG.md`](../AI_USE_LOG.md).

**Key principle:** AI was used as a productivity aid, not as an authority. Every output was reviewed, modified, and validated.

---
## 2. Automated Testing

We wrote **39 automated tests** across 4 test files using pytest:

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', '../tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

### Test Coverage Summary

| Test File | Module Tested | Tests | Coverage |
|-----------|--------------|-------|----------|
| `test_root_finding.py` | `numerical_methods.py` — bisection, Newton-Raphson, secant | 11 | Convergence, accuracy, edge cases, history tracking |
| `test_integration.py` | `numerical_methods.py` — trapezoidal, Simpson's rule | 9 | Constant, linear, quadratic, sine integrals, accuracy comparison |
| `test_linear_systems.py` | `numerical_methods.py` — Gaussian elimination, LU | 7 | 2×2, 3×3, NumPy verification, irrigation problem |
| `test_simulation.py` | `simulation.py`, `data_cleaning.py` | 12 | ET, Euler/RK4, Monte Carlo, outlier detection |

---
## 3. Reproducibility Checklist

In [ ]:
import os

checklist = [
    ('README.md', '../README.md'),
    ('requirements.txt', '../requirements.txt'),
    ('pyproject.toml', '../pyproject.toml'),
    ('AI_USE_LOG.md', '../AI_USE_LOG.md'),
    ('Raw data: weather_daily.csv', '../data/raw/weather_daily.csv'),
    ('Raw data: soil_sensor_data.csv', '../data/raw/soil_sensor_data.csv'),
    ('Raw data: crop_zone_parameters.csv', '../data/raw/crop_zone_parameters.csv'),
    ('src/data_cleaning.py', '../src/data_cleaning.py'),
    ('src/numerical_methods.py', '../src/numerical_methods.py'),
    ('src/simulation.py', '../src/simulation.py'),
    ('src/optimization.py', '../src/optimization.py'),
    ('Level 1 notebook', '../notebooks/Level_1_Problem_Framing.ipynb'),
    ('Level 2 notebook', '../notebooks/Level_2_Vectorization_and_Error.ipynb'),
    ('Level 3 notebook', '../notebooks/Level_3_Numerical_Methods.ipynb'),
    ('Level 4 notebook', '../notebooks/Level_4_Data_Analysis_and_Visualization.ipynb'),
    ('Level 5 notebook', '../notebooks/Level_5_Simulation_and_Optimization.ipynb'),
    ('Level 6 notebook', '../notebooks/Level_6_Final_Integration.ipynb'),
    ('tests/test_root_finding.py', '../tests/test_root_finding.py'),
    ('tests/test_integration.py', '../tests/test_integration.py'),
    ('tests/test_linear_systems.py', '../tests/test_linear_systems.py'),
    ('tests/test_simulation.py', '../tests/test_simulation.py'),
]

print(f"{'Item':<45} {'Status'}")
print('=' * 55)
for name, path in checklist:
    exists = os.path.exists(path)
    print(f"{name:<45} {'PRESENT' if exists else 'MISSING'}")

---
## 4. Full Pipeline Validation

Run the entire pipeline end-to-end to verify reproducibility:

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from src.data_cleaning import load_datasets, clean_weather, clean_soil, save_cleaned_dataset
from src.simulation import compute_et, euler_simulate, rk4_simulate, monte_carlo_rainfall
from src.optimization import greedy_irrigation_schedule
from src.numerical_methods import bisection, trapezoidal_rule, gaussian_elimination

# Step 1: Load and clean
weather_raw, soil_raw, params = load_datasets()
weather, w_log = clean_weather(weather_raw)
soil, s_log = clean_soil(soil_raw)
print(f'1. Data loaded and cleaned ({len(w_log)} weather fixes, {len(s_log)} soil fixes)')

# Step 2: Compute ET
et = compute_et(weather['temperature_c'].values, weather['wind_speed_mps'].values,
                weather['solar_index'].values, weather['humidity_pct'].values)
print(f'2. ET computed: mean={et.mean():.2f}, range=[{et.min():.2f}, {et.max():.2f}]')

# Step 3: Simulate
rainfall = weather['rainfall_mm'].values
zone_a = params[params['zone_id'] == 'Zone_A'].iloc[0]
S0 = soil[soil['zone_id'] == 'Zone_A'].sort_values('timestamp')['soil_moisture_pct'].iloc[0]
S_euler = euler_simulate(S0, rainfall, et, zone_a['drainage_coefficient'], zone_a['field_capacity_pct'])
S_rk4 = rk4_simulate(S0, rainfall, et, zone_a['drainage_coefficient'], zone_a['field_capacity_pct'])
print(f'3. Simulation: Euler final={S_euler[-1]:.2f}%, RK4 final={S_rk4[-1]:.2f}%')

# Step 4: Monte Carlo
scenarios = monte_carlo_rainfall(rainfall, n_scenarios=1000)
print(f'4. Monte Carlo: {scenarios.shape[0]} scenarios generated')

# Step 5: Optimize
irr, moist = greedy_irrigation_schedule(
    S0, rainfall, et, zone_a['drainage_coefficient'],
    zone_a['field_capacity_pct'], zone_a['min_moisture_pct'], zone_a['target_moisture_pct']
)
stress_days = np.sum(moist[1:] < zone_a['min_moisture_pct'])
print(f'5. Optimization: {irr.sum():.1f} mm total irrigation, {stress_days} stress days')

# Step 6: Numerical methods spot check
def f(x): return x**2 - 4
r = bisection(f, 0, 3)
print(f'6. Bisection root of x²-4: {r["root"]:.6f} (expected 2.0), converged={r["converged"]}')

print('\n=== Full pipeline validated successfully ===')

---
## 5. Project Statistics

In [ ]:
import glob

def count_lines(pattern):
    total = 0
    files = glob.glob(pattern, recursive=True)
    for f in files:
        try:
            with open(f) as fh:
                total += sum(1 for _ in fh)
        except:
            pass
    return total, len(files)

src_lines, src_files = count_lines('../src/*.py')
test_lines, test_files = count_lines('../tests/test_*.py')

stats = [
    ('Source modules (src/)', f'{src_files} files, {src_lines} lines'),
    ('Test files (tests/)', f'{test_files} files, {test_lines} lines'),
    ('Notebooks', '6 notebooks (Levels 1-6)'),
    ('Datasets', '3 raw CSVs + 1 cleaned CSV'),
    ('Numerical methods', '12 functions implemented from scratch'),
    ('Automated tests', '39 tests across 4 files'),
    ('Visualizations', '15+ scientific plots'),
    ('Monte Carlo scenarios', '1000 rainfall simulations'),
    ('AI-assisted tasks', '5 (all validated and modified)'),
]

print(f"{'Metric':<35} {'Value'}")
print('=' * 60)
for metric, value in stats:
    print(f'{metric:<35} {value}')

---
## 6. Code Audit Preparation

### Questions We Can Defend

1. **Why bisection over Newton-Raphson?** Bisection is guaranteed to converge if the bracket is valid. Newton-Raphson is faster but requires a derivative and can diverge with a bad initial guess.

2. **Why replace the 45.8°C temperature with median instead of interpolation?** It is a single-point sensor error, not a gap. The median of the remaining data is more representative than interpolating between adjacent days which could be influenced by weather patterns.

3. **Why keep the 85 mm rainfall?** Extreme rainfall events do occur in Kenya during the long rains. Without external evidence that this is a sensor error, removing it would bias the analysis.

4. **Why greedy optimization over gradient descent?** The greedy approach is transparent, interpretable, and guaranteed to find a feasible solution. Gradient descent is available as an alternative but can get stuck in local minima for non-convex constraints.

5. **How do you know the numerical methods are correct?** Every method is validated against: (a) known analytical solutions, (b) NumPy/SciPy reference implementations, and (c) automated pytest assertions.

6. **What happens if sensors fail?** The data cleaning module detects CHECK-status readings and interpolates from neighboring values. The Monte Carlo framework accounts for uncertainty by simulating 1000 scenarios.

---
*End of Level 6 — AI-Assisted Scientific Programming, Reproducibility, Testing, and Presentation.*

*End of HydroSense-Kenya Capstone Project.*